# Hour 3 — Tasks, Forms, Work Packages & Reusable Patterns

Recap: Hour 1 covered auth/projects/companies/users, Hour 2 covered file areas/folders/files. This hour
covers the project-management side of Dalux Build — tasks, forms, work packages — plus a quick tour of the
remaining read-only resource groups, the lower-level utilities the whole package is built on, and a small
capstone that ties everything together.


## 0. Reconnect

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

# Works whether Jupyter was launched from the repo root or from tutorials/
for candidate in (Path(".env"), Path("../.env")):
    if candidate.exists():
        load_dotenv(candidate)
        break
else:
    load_dotenv()  # fall back to variables already exported in the shell

import os
assert os.getenv("DALUX_API_KEY"), "DALUX_API_KEY not found — copy .env.example to .env and fill it in"
assert os.getenv("DALUX_BASE_URL"), "DALUX_BASE_URL not found — copy .env.example to .env and fill it in"
print("DALUX_BASE_URL:", os.getenv("DALUX_BASE_URL"))

In [ ]:
from dalux_build import create_client

dalux = create_client()
dalux

In [ ]:
# Pick the first project in your account to work with for the rest of this notebook.
# Swap this for dalux.projects.get_project_by_name("Your Project Name") if you want a specific one.
projects_response = dalux.projects.list_projects()

PROJECT_ID = projects_response[0].project_id
print("Using project:", projects_response[0].project_name, f"({PROJECT_ID})")

dalux.set_default_project(PROJECT_ID)

## 1. Tasks — `dalux.tasks`

"Tasks" is Dalux Build's umbrella term for issues, approvals, safety observations and good-practice entries. Their
schema varies a lot by task *type*, so the `Task` model is intentionally loose (`extra="allow"`) — it only
guarantees a `task_id`, and passes every other field straight through. Use `.model_dump(by_alias=True)` to see the
raw API field names (`title`, `status`, `deadline`, custom fields, …).

In [ ]:
tasks_response = dalux.tasks.get_project_tasks(to_dataframe=True)
print(f"First page: {len(tasks_response)} task(s)")

Filter by task type with the `typeId` shorthand — the client translates it into the OData filter the API
expects (`$filter=data/type/typeId eq '<typeId>'`) so you don't have to hand-write OData:

In [ ]:
# Discover a real typeId from the first page's raw data before filtering:
if not tasks_response.empty and "type" in tasks_response.columns:
    sample_type_id = tasks_response.iloc[0]["type"].get("typeId") if isinstance(tasks_response.iloc[0]["type"], dict) else None
    print("Example typeId:", sample_type_id)

    filtered = dalux.tasks.get_project_tasks(params={"typeId": sample_type_id}, to_dataframe=True)
    print(f"{len(filtered)} task(s) of that type on the first page")

In [ ]:
# Every task across every page (bookmark pagination handled for you):
all_tasks = dalux.tasks.get_all_project_tasks(verbose=True)
print(f"\nTotal tasks: {len(all_tasks)}")

In [ ]:
if all_tasks:
    task_detail = dalux.tasks.get_task(task_id=all_tasks[0].task_id)

In [ ]:
task_detail.data

## 2. Task change history — `get_project_task_changes` / `get_all_project_task_changes`

A change feed (who changed what, and when) — handy for building an activity log or syncing changes since your
last check.

In [ ]:
changes = dalux.tasks.get_all_project_task_changes(to_dataframe=True)
changes

## 3. Task attachments — `get_project_task_attachments`

In [ ]:
attachments = dalux.tasks.get_project_task_attachments(to_dataframe=True)
attachments

## 4. Forms — `dalux.forms`

Forms are structured records (checklists, reports, …) similar in spirit to tasks.

In [ ]:
forms_response = dalux.forms.get_project_forms(to_dataframe=True)
forms_response

## 5. Work packages — `dalux.work_packages`

In [ ]:
work_packages = dalux.work_packages.list_work_packages(to_dataframe=True)
work_packages

## 6. Quick tour of the remaining read-only resources

These follow the exact same `list_x(project_id, params=None)` shape you've now seen a dozen times.

In [ ]:
print("Inspection plans:", dalux.inspection_plans.list_inspection_plans(to_dataframe=True))

In [ ]:
print("Test plans:", dalux.test_plans.list_test_plans(to_dataframe=True))

In [ ]:
print("Version sets:", dalux.version_sets.get_version_sets(to_dataframe=True))

In [ ]:
print("Project templates (company-wide, no project_id):", dalux.project_templates.list_project_templates())

# Expected to have an "Authentication failed" error.

## 7. Reusable patterns underneath every API class

Two small utilities power almost everything above and are exported from the top-level package so you can reuse
them against **any** endpoint — including ones `dalux_build` doesn't wrap yet:

- `find_by_field(items, field, value)` / `find_all_by_field(...)` — search a list of Pydantic models *or* raw
  dicts by field name, without caring which shape you got back
- `paginate(endpoint, client, params, verbose=True)` — follows `links: [{"rel": "nextPage", ...}]` /
  `bookmark` pagination for you, given a raw endpoint path and the low-level `ApiClient`


In [ ]:
from dalux_build import find_by_field, find_all_by_field

# find_by_field works the same whether you hand it typed Project models or raw dicts
match = find_by_field(projects_response, "project_name", projects_response[0].project_name)
print(match)

## Recap — Hours 1–3

- **Hour 1:** credentials, `create_client()`, the `DaluxClient` namespaces, projects/companies/users, error handling
- **Hour 2:** file areas → folders → files, tree building, downloads (single/bulk/filtered), chunked uploads
- **Hour 3:** tasks, task change history, forms, work packages, the other read-only resources, and the
 `find_by_field` utilities that generalize to any endpoint

For the full method-by-method reference, see the `dalux_build` package's own README (or ask Claude Code — this
repo includes a `dalux-build` skill under `.claude/skills/` that documents every API class and method for
quick lookups while you build).

**Next up — Hour 4:** turning the DataFrames you now know how to fetch into charts with `matplotlib`, and
combining them into a small project dashboard.
